<a href="https://colab.research.google.com/github/oojobe/FB-/blob/main/Traffic_EDA_Group_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Exploratory Data Analysis

## Basic Statistics and Group Analysis

**File:** `synthetic_traffic_benchmark_200k_cleaned.csv` &nbsp;|&nbsp; **Rows:** ~199K &nbsp;|&nbsp; **Period:** 2022-01-01 to 2025-12-31

**How this notebook is organized** (each section states the question being asked, then answers it with code and Reasoning/Findings):
1. Data overview & quality check
2. Basic statistics
3. Road & area context (congestion by area/time)
4. Weather, road conditions & speed compliance
5. Accident Rate by Different Contexts (Road Type, Weather, Time of Day, Area Type)
6. Vehicle Type: Frequency Risk vs. Severity Risk
7. Emergency-response benchmark gap (accidents only, n = 8,213)
8. Supporting Cuts (Driver experience, Holiday season, Average road closures)
9. Key Findings and Summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('synthetic_traffic_benchmark_200k_cleaned.csv', parse_dates=['Date'])

print("Shape:", df.shape)
print("\nColumns and dtypes:")
print(df.dtypes)

Shape: (199403, 30)

Columns and dtypes:
Record_ID                                      object
Timestamp                                      object
Date                                   datetime64[ns]
Hour                                            int64
Month                                           int64
Day_of_Week                                    object
Time_of_Day                                    object
Is_Holiday                                      int64
Area_Type                                      object
Road_Type                                      object
Intersection_Type                              object
Speed_Limit_kmh                                 int64
Weather                                        object
Road_Condition                                 object
Visibility_m                                  float64
Congestion_Index                              float64
Vehicle_Type                                   object
Driver_Age                               

**Reasoning:** before asking any analytical question, we need to know what we're working with — how many
records, what each column represents, and what type each one is (numeric vs categorical vs datetime). This
shapes every choice that follows: which columns can be averaged, which can only be counted, and which one
(`Date`) lets us look at trends over time.

In [ ]:
print("First 5 rows:")
print(df.head().to_string())

First 5 rows:
     Record_ID                      Timestamp       Date  Hour  Month Day_of_Week   Time_of_Day  Is_Holiday Area_Type   Road_Type Intersection_Type  Speed_Limit_kmh Weather Road_Condition  Visibility_m  Congestion_Index Vehicle_Type  Driver_Age  Driver_Experience_yrs  Vehicle_Age_yrs  Recorded_Speed_kmh  Baseline_Expected_Response_Time_min  Accident_Occurred Collision_Type  Vehicles_Involved Accident_Severity Dispatch_Priority  Actual_Response_Time_min  Road_Closure_Duration_min  Year
0  REC_0000000  2024-03-14 17:35:06.841829282 2024-03-14    17      3    Thursday  Evening Peak           0     Urban  Interstate   No Intersection               90   Clear            Dry       10905.0             100.0          SUV        40.0                   16.0              7.2                49.0                                  6.2                  0    No Accident                  0       No Accident    Not Applicable                       0.0                        0.0  2024
1  REC

In [ ]:
# Data quality check
print("Missing values per column (non-zero only):")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None — no missing values.")

print("\nDuplicate Record_IDs:", df['Record_ID'].duplicated().sum())
print("Fully duplicate rows:", df.duplicated().sum())

acc = df[df['Accident_Occurred'] == 1].copy()
acc['Response_Gap_min'] = acc['Actual_Response_Time_min'] - acc['Baseline_Expected_Response_Time_min']

print(f"Loaded {len(df):,} records | {len(acc):,} accidents ({len(acc)/len(df)*100:.2f}%)")

Missing values per column (non-zero only):
None — no missing values.

Duplicate Record_IDs: 0
Fully duplicate rows: 0
Loaded 199,403 records | 8,213 accidents (4.12%)


**Reasoning:** we verify data quality even after cleaning to ensure data wrangling was done properly. Result: **no
missing values and no duplicates**, so we can proceed straight to analysis. We also confirm the accident rate
(**4.12%**) here because it explains why several sections below split into "all records" analysis vs.
"accidents only" analysis — the accident-specific fields are only populated for that 4.12% subset.

## 2. Basic Statistics

**Question:** What are the summary statisticss for numerical columns: count, mean, std, min, 25%, 50%, 75%, max. This should help to understand distribution, central tendency, spread, detect anomalies/outliers, missing values, scale, compare traffic volume/speed

In [ ]:
# Basic statistics on the dataset
print("Basic statictics of traffice dataset:")
print(df.describe(include='all'))

Basic statictics of traffice dataset:
          Record_ID                      Timestamp                           Date           Hour          Month  \
count        199403                         199403                         199403  199403.000000  199403.000000   
unique       199403                         199403                            NaN            NaN            NaN   
top     REC_0000000  2024-03-14 17:35:06.841829282                            NaN            NaN            NaN   
freq              1                              1                            NaN            NaN            NaN   
mean            NaN                            NaN  2024-01-02 12:39:03.033956352      12.223141       6.537484   
min             NaN                            NaN            2022-01-01 00:00:00       0.000000       1.000000   
25%             NaN                            NaN            2023-01-02 00:00:00       8.000000       4.000000   
50%             NaN                       

**Reasoning:** Before building models or visualizations, we need a quick compact numerical overview of the dataset such as congestion index, visisbility, driver details etc. It shows count, mean, min, max, std and others which help identify missing valiues, outliers, unusual ranges and overall distribution of each feature.

## 3. Road & Area Context — Congestion

**Question:** How does congestion differ by area type and time of day? Instead of looking at `Congestion_Index` on its own, we split it by category and compare the distributions, which is far more informative than a single overall average.

In [ ]:
print("Mean Congestion Index by Area_Type:")
print(df.groupby('Area_Type')['Congestion_Index'].mean().round(1).sort_values(ascending=False))
print("\nMean Congestion Index by Time_of_Day:")
print(df.groupby('Time_of_Day', observed=True)['Congestion_Index'].mean().round(1))

Mean Congestion Index by Area_Type:
Area_Type
Urban       67.8
Suburban    53.9
Rural       34.6
Name: Congestion_Index, dtype: float64

Mean Congestion Index by Time_of_Day:
Time_of_Day
Evening Peak    69.8
Midday          49.4
Morning Peak    69.8
Night           20.0
Name: Congestion_Index, dtype: float64


**Findings:** congestion scales cleanly with density — **Urban (67.8) > Suburban (53.9) > Rural (34.6)** — and with commute timing — **Morning/Evening Peak (≈69.8 each) far exceed Midday (49.4) and Night (20.0)**. Both results move in the expected direction based on community distribution and work hours.

## 4. Weather, Road Conditions & Speed Compliance

**Question:** Do drivers actually slow down in bad weather, and how often is the speed limit exceeded? We compare `Recorded_Speed_kmh` against `Speed_Limit_kmh` (their difference, `Speed_Diff`) across weather categories.

In [ ]:
df['Speed_Diff'] = df['Recorded_Speed_kmh'] - df['Speed_Limit_kmh']
weather_order = ['Clear','Fog','Rain','Snow']

data_by_weather = [df.loc[df['Weather']==w, 'Speed_Diff'].values for w in weather_order]

pct_over = df.groupby('Weather')['Speed_Diff'].apply(lambda s: (s > 0).mean() * 100).reindex(weather_order)

print("Mean/median speed vs. limit (km/h) by weather:")
print(df.groupby('Weather')['Speed_Diff'].agg(['mean','median']).reindex(weather_order).round(2))
print("\n% exceeding limit by weather:")
print(pct_over.round(1))

Mean/median speed vs. limit (km/h) by weather:
          mean  median
Weather               
Clear    -5.52    -3.4
Fog      -6.25    -4.1
Rain     -9.51    -6.9
Snow    -22.00   -18.8

% exceeding limit by weather:
Weather
Clear    30.8
Fog      27.6
Rain     16.2
Snow      0.1
Name: Speed_Diff, dtype: float64


**Findings:** In clear weather, drivers run close to (and often above) the limit — **30.8% exceed it**, averaging only 5.5 km/h under on the whole. In snow, that drops
to essentially **0.1% exceeding the limit**, with drivers averaging **22 km/h under** the posted limit. Rain and fog sit in between. Drivers in this dataset are behaving exactly as you'd want them to — slowing down as conditions worsen — which is worth keeping in mind when we look at accident rates by weather next: higher accident rates in bad weather are happening *despite* slower speeds, not because of extra speeding.

## 5. Accident Rate by Different Contexts

**Question:** What are the accident rates for the Road type, Weather, Time of day, Area type and Intersection types? Are these figures expected or skew from normal? What type of accidents are mostly seen what is their severity?

In [ ]:
for col in ['Road_Type', 'Weather', 'Time_of_Day', 'Area_Type', 'Intersection_Type']:
    rates = (df.groupby(col, observed=True)['Accident_Occurred'].mean() * 100).round(2).sort_values(ascending=False)
    print(f"Accident rate % by {col}:")
    print(rates.to_string())
    print()

Accident rate % by Road_Type:
Road_Type
Interstate    7.93
Arterial      4.46
Collector     2.85
Local         2.01

Accident rate % by Weather:
Weather
Snow     5.93
Rain     4.98
Fog      3.78
Clear    3.48

Accident rate % by Time_of_Day:
Time_of_Day
Morning Peak    5.20
Evening Peak    5.19
Midday          3.40
Night           1.88

Accident rate % by Area_Type:
Area_Type
Urban       4.57
Suburban    3.87
Rural       3.71

Accident rate % by Intersection_Type:
Intersection_Type
No Intersection    5.25
Roundabout         3.75
4-Way              3.70
T-Junction         3.24



**Findings:** Road_Type shows the widest spread of any single factor where Interstate (7.93%) is roughly 4x Local (2.01%). Weather and Time of Day move in the expected direction (worse conditions / peak hours → higher accident rates) and Area Type shows the mildest spread of the four.

In [ ]:
sev_order = ['Minor','Moderate','Severe','Fatal']
coll_order = ['Rear-End','Side-Impact','Single-Vehicle','Head-On']
sev_counts = acc['Accident_Severity'].value_counts().reindex(sev_order)
coll_counts = acc['Collision_Type'].value_counts().reindex(coll_order)

print("Severity breakdown:")
print(sev_counts)
print("\nCollision type breakdown:")
print(coll_counts)

Severity breakdown:
Accident_Severity
Minor       5366
Moderate    2351
Severe       348
Fatal        148
Name: count, dtype: int64

Collision type breakdown:
Collision_Type
Rear-End          4893
Side-Impact       1591
Single-Vehicle    1488
Head-On            241
Name: count, dtype: int64


**Findings:** the large majority of accidents are low-severity with **65.3% Minor**, 28.6% Moderate, and only 4.2% Severe / 1.8% Fatal combined. Rear-end collisions dominate (59.6%), which fits the earlier finding that Interstates and peak-hour congestion carry the most accidents. Rear-end collisions are the classic outcome of dense, stop-and-go highway traffic.

## 6. Vehicle Type: Frequency Risk vs. Severity Risk

**Question:** What vehicle type is most commonly associated with accidents? What are the severity levels of these accidents?

In [ ]:
veh_rate = (df.groupby('Vehicle_Type')['Accident_Occurred'].mean() * 100).round(2)
sev_fatal_share = acc.groupby('Vehicle_Type')['Accident_Severity'].apply(
    lambda s: s.isin(['Severe', 'Fatal']).mean() * 100
).round(2)

vehicle_summary = pd.DataFrame({
    'Accident_Rate_%': veh_rate,
    'Severe_or_Fatal_Share_%': sev_fatal_share
}).sort_values('Severe_or_Fatal_Share_%', ascending=False)

print(vehicle_summary)

              Accident_Rate_%  Severe_or_Fatal_Share_%
Vehicle_Type                                          
Motorcycle               4.36                    39.04
Truck                    3.59                    28.07
Bus                      3.94                    22.94
SUV                      4.04                     0.95
Sedan                    4.27                     0.77


**Findings:** Accident rates barely differs by vehicle (3.6–4.4% across the board) with motorcycles accounting for the highest, but the *severity* of those accidents differ enormously where Motorcycle accidents are severe/fatal 39.0% of the time vs well under 1% for Sedan/SUV. Frequency and severity are answering two different questions here.

## 7. Emergency-Response Gap (accidents only, n = 8,213)

**Question:** What is the overall emergency response time average? What factors affect the rate of the emergency response time *increase or decrease*?

In [ ]:
print(f"Overall mean response gap: {acc['Response_Gap_min'].mean():.2f} min")
print(f"Share of accidents where actual exceeded baseline: {(acc['Response_Gap_min']>0).mean()*100:.1f}%")
print()

for col in ['Area_Type', 'Weather', 'Accident_Severity', 'Dispatch_Priority', 'Vehicles_Involved']:
    g = acc.groupby(col, observed=True)['Response_Gap_min'].agg(['mean', 'count']).round(2).sort_values('mean', ascending=False)
    print(f"Mean response gap (min) by {col}:")
    print(g.to_string())
    print()

Overall mean response gap: 9.94 min
Share of accidents where actual exceeded baseline: 97.8%

Mean response gap (min) by Area_Type:
            mean  count
Area_Type              
Rural      12.99   1470
Urban       9.75   3657
Suburban    8.72   3086

Mean response gap (min) by Weather:
          mean  count
Weather              
Fog      14.58    669
Snow     12.55   1363
Rain      9.23   2185
Clear     8.67   3996

Mean response gap (min) by Accident_Severity:
                    mean  count
Accident_Severity              
Minor              10.73   5366
Moderate            8.50   2351
Fatal               8.48    148
Severe              8.12    348

Mean response gap (min) by Dispatch_Priority:
                    mean  count
Dispatch_Priority              
Standard           11.83   4115
High                8.07   2550
Medium              8.00   1548

Mean response gap (min) by Vehicles_Involved:
                    mean  count
Vehicles_Involved              
1                  11.

**Findings:** the baseline is missed in 97.8% of accidents by an average of 9.94 minutes. The gap is worst in Rural areas and during Foggy weather, and a more encouraging pattern is the smaller response time for higher-severity accidents
and higher dispatch priorities, suggesting triage works directionally even though the target is missed almost everywhere. Single-vehicle accidents have the *largest* gap of any `Vehicles_Involved` group (11.86 min), which cuts against the assumption that more complex scenes get slower response.

## 8. Supporting Cuts

**Questions:** Do accidents decrease the more experienced or older drivers get? Does the holiday season affect the congestion time and accident rate? What are the average road closure time for each severity of accidents?

In [ ]:
def to_hours_mins(minutes):
    if pd.isna(minutes):
        return "NaN"
    h = int(minutes // 60)
    m = int(round(minutes % 60))
    return f"{h}h {m}m" if h > 0 else f"{m}m"

age_bins = [15,25,35,45,55,65,90]
age_labels = ['16-25','26-35','36-45','46-55','56-65','66+']
df['Age_Bucket'] = pd.cut(df['Driver_Age'], bins=age_bins, labels=age_labels)
age_rates = (df.groupby('Age_Bucket', observed=True)['Accident_Occurred'].mean() * 100).reindex(age_labels)
print("Accident rate % by driver age:")
print(age_rates.round(2))

exp_bins = [-1, 2, 5, 10, 20, 60]
exp_labels = ['0-2', '3-5', '6-10', '11-20', '20+']
df['Exp_Bucket'] = pd.cut(df['Driver_Experience_yrs'], bins=exp_bins, labels=exp_labels)
print("Accident rate % by driver experience:")
print((df.groupby('Exp_Bucket', observed=True)['Accident_Occurred'].mean() * 100).round(2).to_string())

print("\nHoliday effect:")
print("  Accident rate % — Holiday:", round(df.loc[df['Is_Holiday']==1,'Accident_Occurred'].mean()*100, 2),
      "| Regular day:", round(df.loc[df['Is_Holiday']==0,'Accident_Occurred'].mean()*100, 2))
print("  Mean congestion — Holiday:", round(df.loc[df['Is_Holiday']==1,'Congestion_Index'].mean(), 1),
      "| Regular day:", round(df.loc[df['Is_Holiday']==0,'Congestion_Index'].mean(), 1))

print("\nMean road closure duration (min) by severity:")
print(acc.groupby('Accident_Severity', observed=True)['Road_Closure_Duration_min'].agg(['mean','median','count']).reindex(sev_order).round(0).to_string())
summary = acc.groupby('Accident_Severity', observed=True)['Road_Closure_Duration_min'].agg(['mean','median','count']).reindex(sev_order).round(0)
summary['mean'] = summary['mean'].apply(to_hours_mins)
summary['median'] = summary['median'].apply(to_hours_mins)
print("\nMean road closure duration (hr) by severity:")
print(summary.to_string())


Accident rate % by driver age:
Age_Bucket
16-25    4.73
26-35    4.04
36-45    4.03
46-55    3.97
56-65    3.73
66+      3.74
Name: Accident_Occurred, dtype: float64
Accident rate % by driver experience:
Exp_Bucket
0-2      4.98
3-5      4.43
6-10     3.98
11-20    3.89
20+      3.73

Holiday effect:
  Accident rate % — Holiday: 3.14 | Regular day: 4.15
  Mean congestion — Holiday: 43.9 | Regular day: 56.0

Mean road closure duration (min) by severity:
                    mean  median  count
Accident_Severity                      
Minor              121.0   117.0   5366
Moderate           241.0   232.0   2351
Severe             484.0   484.0    348
Fatal              959.0   903.0    148

Mean road closure duration (hr) by severity:
                      mean  median  count
Accident_Severity                        
Minor                2h 1m  1h 57m   5366
Moderate             4h 1m  3h 52m   2351
Severe               8h 4m   8h 4m    348
Fatal              15h 59m  15h 3m    148


**Findings:** The youngest bracket (16–25) has the highest accident rate at **4.73%**, above the overall average of 4.12%, and the rate declines fairly steadily with age before flattening out around 3.7–3.9% for drivers over 35. The gradient is real but mild. Accident rates also decline steadily with driver experience (4.98% → 3.73%). Holidays run **lower** on both accident rate and congestion than regular days which is consistent with lighter commuter traffic.
Road closure duration scales almost linearly with severity, roughly 8x from Minor (121 min) to Fatal (959 min).
Therefore age/experience is a contributing factor to accident rates, but not a dominant one compared to road type or weather.

## 9. Key Findings and Summary

**Traffic conditions**
- Congestion tracks density and commute timing cleanly: Urban > Suburban > Rural; Peak hours ≈ 70 vs. Night ≈ 20.
- Drivers self-regulate speed by weather: 30.8% exceed the limit in clear weather vs. 0.1% in snow.

**Accidents (4.12% of all records)**
- Road type is the single strongest split: Interstate (7.93%) vs. Local (2.01%) — a 4x gap.
- Snow, Rain and Fog all raise accident rate above Clear, *despite* drivers slowing down more in those conditions.
- Peak-hour accident rate (~5.2%) is nearly 3x the Night rate (1.88%).
- Most accidents are low-severity (65.3% Minor) and rear-end collisions dominate (59.6%).
- Youngest drivers (16–25) have the highest accident rate (4.73%), tapering with age.

**Emergency-response benchmark — the central question**
- Actual response time misses the baseline in 97.8% of accidents, averaging 9.94 minutes late (baseline 12.0 min → actual 22.0 min).
- The gap is worst in Rural areas and in Fog (14.6 min average gap).
- Encouragingly, higher-severity accidents and higher dispatch priorities show *smaller* gaps — the triage
  system appears to work directionally, even though the baseline is missed almost everywhere.
- Road closure duration rises roughly 8x from Minor to Fatal accidents (121 min → 959 min).